In [76]:
import pandas as pd
from pathlib import Path

# Paths
DATA = Path('..') / 'data'

# Load and combine all ATP ranking files
atp_rankings = pd.concat([
    pd.read_csv(DATA / 'atp_rankings_00s.csv'),
    pd.read_csv(DATA / 'atp_rankings_10s.csv'),
    pd.read_csv(DATA / 'atp_rankings_20s.csv'),
    pd.read_csv(DATA / 'atp_rankings_current.csv'),
], ignore_index=True)

# Same for WTA
wta_rankings = pd.concat([
    pd.read_csv(DATA / 'wta_rankings_00s.csv'),
    pd.read_csv(DATA / 'wta_rankings_10s.csv'),
    pd.read_csv(DATA / 'wta_rankings_20s.csv'),
    pd.read_csv(DATA / 'wta_rankings_current.csv'),
], ignore_index=True)

# Load player metadata
atp_players = pd.read_csv(DATA / 'atp_players.csv')
wta_players = pd.read_csv(DATA / 'wta_players.csv')

print(f"ATP rankings rows: {len(atp_rankings):,}")
print(f"WTA rankings rows: {len(wta_rankings):,}")
print(f"ATP players: {len(atp_players):,}")
print(f"WTA players: {len(wta_players):,}")

# Quick peek
atp_rankings.head()

ATP rankings rows: 2,382,667
WTA rankings rows: 1,647,451
ATP players: 66,820
WTA players: 70,498


C:\Users\endig\AppData\Local\Temp\ipykernel_44748\1799983553.py:24: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  atp_players = pd.read_csv(DATA / 'atp_players.csv')
C:\Users\endig\AppData\Local\Temp\ipykernel_44748\1799983553.py:25: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  wta_players = pd.read_csv(DATA / 'wta_players.csv')


,ranking_date,rank,player,points
0,20000110,1,101736,4135.0
1,20000110,2,102338,2915.0
2,20000110,3,101948,2419.0
3,20000110,4,103017,2184.0
4,20000110,5,102856,2169.0


In [77]:
# Convert ranking_date from integer (YYYYMMDD) to actual datetime
atp_rankings['ranking_date'] = pd.to_datetime(
    atp_rankings['ranking_date'], format='%Y%m%d'
)
wta_rankings['ranking_date'] = pd.to_datetime(
    wta_rankings['ranking_date'], format='%Y%m%d'
)

# Verify the conversion worked
print(atp_rankings['ranking_date'].min(), '→', atp_rankings['ranking_date'].max())
print(wta_rankings['ranking_date'].min(), '→', wta_rankings['ranking_date'].max())
atp_rankings.head()

2000-01-10 00:00:00 → 2026-05-04 00:00:00
2000-01-01 00:00:00 → 2026-05-04 00:00:00


,ranking_date,rank,player,points
0,2000-01-10,1,101736,4135.0
1,2000-01-10,2,102338,2915.0
2,2000-01-10,3,101948,2419.0
3,2000-01-10,4,103017,2184.0
4,2000-01-10,5,102856,2169.0


In [78]:
# Combine first and last name into a single column for readability
atp_players['name'] = atp_players['name_first'] + ' ' + atp_players['name_last']
wta_players['name'] = wta_players['name_first'] + ' ' + wta_players['name_last']

# Add gender column for when we eventually combine the two
atp_players['gender'] = 'M'
wta_players['gender'] = 'W'

# Select only the player columns we need
atp_players_slim = atp_players[['player_id', 'name', 'ioc', 'gender']]
wta_players_slim = wta_players[['player_id', 'name', 'ioc', 'gender']]

# Join rankings with player metadata
# Note: rankings uses 'player' as the column name, players uses 'player_id'
atp_merged = atp_rankings.merge(
    atp_players_slim, 
    left_on='player', 
    right_on='player_id', 
    how='left'
)

wta_merged = wta_rankings.merge(
    wta_players_slim,
    left_on='player',
    right_on='player_id',
    how='left'
)

# Quick check — how many rankings have a matching player?
print(f"ATP — unmatched rankings: {atp_merged['name'].isna().sum():,}")
print(f"WTA — unmatched rankings: {wta_merged['name'].isna().sum():,}")

atp_merged.head()

ATP — unmatched rankings: 43
WTA — unmatched rankings: 0


,ranking_date,rank,player,points,player_id,name,ioc,gender
0,2000-01-10,1,101736,4135.0,101736,Andre Agassi,USA,M
1,2000-01-10,2,102338,2915.0,102338,Yevgeny Kafelnikov,RUS,M
2,2000-01-10,3,101948,2419.0,101948,Pete Sampras,USA,M
3,2000-01-10,4,103017,2184.0,103017,Nicolas Kiefer,GER,M
4,2000-01-10,5,102856,2169.0,102856,Gustavo Kuerten,BRA,M


In [79]:
# Filter to Italian players only
atp_ita = atp_merged[atp_merged['ioc'] == 'ITA'].copy()
wta_ita = wta_merged[wta_merged['ioc'] == 'ITA'].copy()

print(f"ATP Italian ranking rows: {len(atp_ita):,}")
print(f"WTA Italian ranking rows: {len(wta_ita):,}")
print(f"Unique Italian men: {atp_ita['name'].nunique()}")
print(f"Unique Italian women: {wta_ita['name'].nunique()}")


ATP Italian ranking rows: 144,906
WTA Italian ranking rows: 82,855
Unique Italian men: 670
Unique Italian women: 365


In [80]:
# Combine ATP and WTA Italian rankings
italian_rankings = pd.concat([atp_ita, wta_ita], ignore_index=True)

# Drop the redundant player_id column (same as 'player')
italian_rankings = italian_rankings.drop(columns=['player_id','tours','ioc'])

# Add a 'year' column for easy Tableau grouping
italian_rankings['year'] = italian_rankings['ranking_date'].dt.year

# Confirm
print(f"Total Italian rankings: {len(italian_rankings):,}")
print(f"Date range: {italian_rankings['ranking_date'].min().date()} → {italian_rankings['ranking_date'].max().date()}")
print(f"Gender breakdown:")
print(italian_rankings['gender'].value_counts())
print()
italian_rankings.head()


Total Italian rankings: 227,761
Date range: 2000-01-01 → 2026-05-04
Gender breakdown:
gender
M    144906
W     82855
Name: count, dtype: int64



,ranking_date,rank,player,points,name,gender,year
0,2000-01-10,81,101150,465.0,Gianluca Pozzi,M,2000
1,2000-01-10,85,102133,448.0,Laurence Tieleman,M,2000
2,2000-01-10,89,102247,440.0,Andrea Gaudenzi,M,2000
3,2000-01-10,112,102106,353.0,Davide Sanguinetti,M,2000
4,2000-01-10,155,102433,244.0,Mose Navarra,M,2000


In [81]:
# Latest ranking date in our data
latest_date = italian_rankings['ranking_date'].max()
print(f"Latest data: {latest_date.date()}")
print()

# Top 5 Italian men currently
print("Top 5 Italian men (most recent):")
print(italian_rankings[
    (italian_rankings['ranking_date'] == latest_date) & 
    (italian_rankings['gender'] == 'M')
].nsmallest(5, 'rank')[['name', 'rank', 'points']].to_string(index=False))
print()

# Top 5 Italian women currently
print("Top 5 Italian women (most recent):")
print(italian_rankings[
    (italian_rankings['ranking_date'] == latest_date) & 
    (italian_rankings['gender'] == 'W')
].nsmallest(5, 'rank')[['name', 'rank', 'points']].to_string(index=False))

Latest data: 2026-05-04

Top 5 Italian men (most recent):
           name  rank  points
  Jannik Sinner     1 14350.0
Lorenzo Musetti    10  3415.0
 Flavio Cobolli    12  2750.0
Luciano Darderi    20  1890.0
 Lorenzo Sonego    66   830.0

Top 5 Italian women (most recent):
                  name  rank  points
       Jasmine Paolini     8  3722.0
Elisabetta Cocciaretto    41  1290.0
           Lisa Pigato   141   547.0
    Lucrezia Stefanini   154   495.0
      Nuria Brancaccio   171   463.0


In [82]:
output_path = DATA / 'italian_rankings.csv'
italian_rankings.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Rows: {len(italian_rankings):,}")

Saved: ..\data\italian_rankings.csv
Rows: 227,761


In [87]:
# For each player-year, find best rank AND max points
year_summary = (
    italian_rankings
    .groupby(['year', 'gender', 'player', 'name'], as_index=False)
    .agg(
        best_rank=('rank', 'min'),
        max_points=('points', 'max')
    )
)
print(year_summary.head())
print(f"Shape: {year_summary.shape}")

   year gender  player            name  best_rank  max_points
0  2000      M  100956  Simone Colombo       1156         2.0
1  2000      M  101150  Gianluca Pozzi         42       854.0
2  2000      M  101481  Omar Camporese        704        16.0
3  2000      M  101717   Diego Nargiso        134       281.0
4  2000      M  101746    Renzo Furlan        221       148.0
Shape: (6065, 6)


In [89]:
# Count distinct players reaching each threshold per (year, gender)
yearly_top_counts = []

for threshold in [10, 50, 100,500]:
    counts = (
        year_summary[year_summary['best_rank'] <= threshold]
        .groupby(['year', 'gender'], as_index=False)
        .size()
        .rename(columns={'size': 'count'})
    )
    counts['threshold'] = f'Top {threshold}'
    yearly_top_counts.append(counts)

yearly_top_counts = pd.concat(yearly_top_counts, ignore_index=True)

# Reorder for clarity
yearly_top_counts = yearly_top_counts[['year', 'gender', 'threshold', 'count']]

# Save
yearly_top_counts.to_csv(DATA / 'yearly_top_counts.csv', index=False)

print(yearly_top_counts.head(10))
print(f"\nShape: {yearly_top_counts.shape}")

   year gender threshold  count
0  2009      W    Top 10      1
1  2010      W    Top 10      2
2  2011      W    Top 10      1
3  2012      W    Top 10      1
4  2013      W    Top 10      1
5  2014      W    Top 10      1
6  2015      W    Top 10      1
7  2016      W    Top 10      2
8  2019      M    Top 10      2
9  2020      M    Top 10      1

Shape: (180, 4)


In [85]:
# Players with the best rank in each (year, gender) group
# Now pick the top Italian per (year, gender): best rank, then most points as tiebreaker
top_italian_by_rank = (
    year_summary.sort_values(
                            ['year', 'gender', 'best_rank', 'max_points'],
                            ascending=[True, True, True, False]  # best rank first, then more points
    )
    .groupby(['year', 'gender'], as_index=False)
    .first()
    [['year', 'gender', 'name', 'best_rank','max_points']]
)
top_italian_by_rank.to_csv(DATA / 'top_italian_by_rank.csv', index=False)


print(top_italian_by_rank.head(20))
print(f"\nShape: {top_italian_by_rank.shape}")

    year gender                 name  best_rank  max_points
0   2000      M       Gianluca Pozzi         42       854.0
1   2000      W   Silvia Farina Elia         26       976.0
2   2001      M       Gianluca Pozzi         40       864.0
3   2001      W   Silvia Farina Elia         14      1743.0
4   2002      M   Davide Sanguinetti         43       781.0
5   2002      W   Silvia Farina Elia         11      1838.0
6   2003      M   Davide Sanguinetti         44       773.0
7   2003      W   Silvia Farina Elia         15      1596.0
8   2004      M     Filippo Volandri         35       890.0
9   2004      W   Silvia Farina Elia         14      1736.0
10  2005      M     Filippo Volandri         28      1140.0
11  2005      W  Francesca Schiavone         13      1704.0
12  2006      M     Filippo Volandri         31       961.0
13  2006      W  Francesca Schiavone         11      1699.0
14  2007      M     Filippo Volandri         25      1091.0
15  2007      W  Francesca Schiavone    

In [58]:
# Alternative definition: top Italian by total accumulated points each year
top_italian_by_points = (
    year_summary
    .sort_values(
        ['year', 'gender', 'max_points', 'best_rank'],
        ascending=[True, True, False, True]   # most points first, best rank as tiebreaker
    )
    .groupby(['year', 'gender'], as_index=False)
    .first()
    [['year', 'gender', 'name', 'max_points', 'best_rank']]
)

# Save
top_italian_by_points.to_csv(DATA / 'top_italian_by_points.csv', index=False)

print(top_italian_by_points.head(20))
print(f"\nShape: {top_italian_by_points.shape}")

    year gender                 name  max_points  best_rank
0   2000      M       Gianluca Pozzi       854.0         42
1   2000      W   Silvia Farina Elia       976.0         26
2   2001      M       Gianluca Pozzi       864.0         40
3   2001      W   Silvia Farina Elia      1743.0         14
4   2002      M   Davide Sanguinetti       781.0         43
5   2002      W   Silvia Farina Elia      1838.0         11
6   2003      M     Filippo Volandri       818.0         46
7   2003      W   Silvia Farina Elia      1596.0         15
8   2004      M     Filippo Volandri       890.0         35
9   2004      W   Silvia Farina Elia      1736.0         14
10  2005      M     Filippo Volandri      1140.0         28
11  2005      W  Francesca Schiavone      1704.0         13
12  2006      M     Filippo Volandri       961.0         31
13  2006      W  Francesca Schiavone      1699.0         11
14  2007      M     Filippo Volandri      1091.0         25
15  2007      W  Francesca Schiavone    

In [86]:
# Compare the two definitions side by side
comparison = top_italian_by_rank[['year', 'gender', 'name']].merge(
    top_italian_by_points[['year', 'gender', 'name']],
    on=['year', 'gender'],
    suffixes=('_by_rank', '_by_points')
)

# Find disagreements
disagreements = comparison[comparison['name_by_rank'] != comparison['name_by_points']]

print(f"Years where the two definitions disagree: {len(disagreements)}")
print()
if len(disagreements) > 0:
    print(disagreements.to_string(index=False))

Years where the two definitions disagree: 4

 year gender       name_by_rank   name_by_points
 2003      M Davide Sanguinetti Filippo Volandri
 2010      M      Andreas Seppi   Potito Starace
 2019      M  Matteo Berrettini    Fabio Fognini
 2022      W   Martina Trevisan    Camila Giorgi


In [91]:
# Compute the best ranking achieved by each player

career_best = (
    italian_rankings
    .groupby(['gender', 'player', 'name'], as_index=False)
    .agg(
        career_best_rank=('rank', 'min'),
        first_ranked=('ranking_date', 'min'),
        last_ranked=('ranking_date', 'max'),
        weeks_ranked=('ranking_date', 'count'),
    )
)

career_best['career_span_years'] = (
    (career_best['last_ranked'] - career_best['first_ranked']).dt.days / 365.25
).round(1)

career_best = career_best.sort_values('career_best_rank').reset_index(drop=True)

career_best.to_csv(DATA / 'career_best.csv', index=False)

print("Top 15 Italian career-bests (both genders):")
print(career_best.head(15).to_string(index=False))
print(f"\nShape: {career_best.shape}")

Top 15 Italian career-bests (both genders):
gender  player                name  career_best_rank first_ranked last_ranked  weeks_ranked  career_span_years
     M  206173       Jannik Sinner                 1   2018-02-12  2026-05-04           351                8.2
     W  201212 Francesca Schiavone                 4   2000-01-01  2018-09-10           970               18.7
     W  211148     Jasmine Paolini                 4   2013-06-17  2026-05-04           622               12.9
     W  201506         Sara Errani                 5   2002-08-26  2026-05-04          1194               23.7
     M  207518     Lorenzo Musetti                 5   2019-04-08  2026-05-04           298                7.1
     M  126610   Matteo Berrettini                 6   2015-04-06  2026-05-04           485               11.1
     W  201355     Flavia Pennetta                 6   2000-01-01  2016-07-04           870               16.5
     W  201311       Roberta Vinci                 7   2000-01-01  2